# MaktspråkAI — retrain the party classifier on a Colab GPU

Run against a **Colab GPU runtime** (colab.research.google.com, or VS Code → *Select Kernel → Colab*).

The runtime is a fresh VM: we mount Drive (for checkpoints that survive disconnects), clone the repo, install deps, and inject credentials. Add these **Colab Secrets** (key icon in the left sidebar, notebook access on):

- `SUPABASE_URL`
- `SUPABASE_KEY`
- `HF_TOKEN` (Hugging Face write token, for publishing the model)

Running from VS Code instead of the Colab UI? Secrets aren't reachable there — cell 2 falls back to pasting the values.

**Prerequisites:** the `fix/pdf-parser` branch is pushed to GitHub, and the Supabase `speeches` table is re-indexed with the fixed parser (`scripts/reindex_speeches.py`).

In [ ]:
# 1. Confirm the GPU and mount Drive (checkpoints survive disconnects there)
!nvidia-smi -L
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# 2. Credentials -> environment. Colab Secrets in the Colab UI; paste-fallback elsewhere.
import os

for key in ("SUPABASE_URL", "SUPABASE_KEY", "HF_TOKEN"):
    try:
        from google.colab import userdata

        os.environ[key] = userdata.get(key)
    except Exception:
        import getpass

        os.environ[key] = getpass.getpass(f"{key}: ")
print("Credentials set:", all(os.environ.get(k) for k in ("SUPABASE_URL", "SUPABASE_KEY", "HF_TOKEN")))

In [ ]:
# 3. Clone the repo and install dependencies
%cd /content
!rm -rf MaktsprakAI
!git clone https://github.com/MartinBlomqvistDev/MaktsprakAI.git
%cd MaktsprakAI
!pip install -q -r requirements.txt

In [ ]:
# 4. Train (full run). Checkpoints + best model land on Drive, so a disconnect
#    just means re-running this cell — it resumes from the last epoch.
#    Cheap baseline pass instead: add --no-fgm --max-length 256
%cd /content/MaktsprakAI
!python scripts/train_party_model_db.py \
    --output-dir /content/drive/MyDrive/MaktsprakAI_Checkpoints

In [ ]:
# 5. Benchmark the new model against the live one on the speaker-independent set
!python scripts/evaluate_model.py \
    --model MartinBlomqvist/maktsprak_classifier_clean \
    --model /content/drive/MyDrive/MaktsprakAI_Checkpoints \
    --val-speakers /content/drive/MyDrive/MaktsprakAI_Checkpoints/val_speakers.json \
    --limit 2000

import glob

from IPython.display import Image, display

for path in sorted(glob.glob("results/confusion_*.png")):
    display(Image(path))

In [ ]:
# 6. If the new model wins the benchmark: push it to a NEW HF repo for A/B.
#    Promote to MartinBlomqvist/maktsprak_classifier_clean only once satisfied —
#    the live app loads that repo, so pushing there deploys immediately.
import os

from huggingface_hub import HfApi

api = HfApi(token=os.environ["HF_TOKEN"])
api.upload_folder(
    folder_path="/content/drive/MyDrive/MaktsprakAI_Checkpoints",
    repo_id="MartinBlomqvist/maktsprak_classifier_reindexed",
    repo_type="model",
    commit_message="Speaker-independent retrain on re-indexed data",
    ignore_patterns=["checkpoints/*", "*.pt"],  # HF export only, no raw checkpoints
)
print("Pushed to MartinBlomqvist/maktsprak_classifier_reindexed")